In [1]:
%load_ext autoreload
%autoreload 2

# Predict with the Flow Matching Model or CGM

In [2]:
import importlib

import hydra
import lightning as L
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import log_scores, update_wandb_run
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

torch.set_float32_matmul_precision("medium")

In [3]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/adb68obp"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
    # Do not shuffle any dataloader
    cfg.data.module.dataloader_config.train.shuffle = False
    cfg.data.module.dataloader_config.val.shuffle = False
    cfg.data.module.dataloader_config.val.batch_size = 16  # For faster predictions
    cfg.data.module.dataloader_config.test.shuffle = False
    datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")

In [5]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

In [6]:
trainer = L.Trainer(logger=False, devices=[1])
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 46/46 [00:03<00:00, 13.37it/s]


In [7]:
# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

In [9]:
# load observations
val_split = hydra.utils.instantiate(cfg.data.splits.val)
y_val = xr.open_dataarray(OBSERVATIONS_FLAT_PATH).sel(time=val_split)
y_t = torch.from_numpy(y_val.values).to(predictions_rescaled)

In [10]:
# Compute the scores
crps_ens = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

crps_per_margin = crps_ens(predictions_rescaled, y_t)

# Rearrange so that we compute the scores seperatly for each variable
x_spatial = rearrange(predictions_rescaled, "t n d lat lon -> t d n (lat lon)")
y_spatial = rearrange(y_t, "t d lat lon -> t d (lat lon)")

energy_score_per_var = es(x_spatial, y_spatial)

# For the VS there are huge intermediary results, thats why it is computed batchwise
vss = []
for x_i, y_i in tqdm(zip(x_spatial, y_spatial), total=predictions_rescaled.shape[0]):
    vss.append(vs(x_i, y_i))
variogram_score_per_var = torch.stack(vss)


# Rearrange to compute scores for both variables together
x_full = rearrange(predictions_rescaled, "t n d lat lon -> t n (d lat lon)")
y_full = rearrange(y_t, "t d lat lon -> t (d lat lon)")

energy_score_full = es(x_full, y_full)

vss = []
for x_i, y_i in tqdm(zip(x_full, y_full), total=predictions_rescaled.shape[0]):
    vss.append(vs(x_i, y_i))
variogram_score_full = torch.stack(vss)

100%|██████████| 730/730 [05:41<00:00,  2.14it/s]


In [11]:
# Reduce the scores
variogram_score_per_var = reduce(variogram_score_per_var, "t d -> d", "mean")
energy_score_per_var = reduce(energy_score_per_var, "t d -> d", "mean")
crps_per_var = reduce(crps_per_margin, "t d h w -> d", reduction="mean")

variogram_score_full = reduce(variogram_score_full, "t -> 1", "mean")
energy_score_full = reduce(energy_score_full, "t -> 1", "mean")
crps_full = reduce(crps_per_margin, "t d h w -> 1", "mean")

In [12]:
# Log the Scores
model_class = cfg.model._target_.split(".")[-1]
# Log scores per Variable
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="CRPS",
    variables=datamodule.y_select_variables,
    scores=crps_per_var,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=energy_score_per_var,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="VariogramScore",
    variables=datamodule.y_select_variables,
    scores=variogram_score_per_var,
)
# Log full scores
log_scores(
    file=SCORE_FILE, model=model_class, metric="CRPS", variables=["combined"], scores=crps_full
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=["combined"],
    scores=energy_score_full,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="VariogramScore",
    variables=["combined"],
    scores=variogram_score_full,
)

In [13]:
# Update WandB run
val_scores = {
    "CRPS_combined": crps_full.item(),
    "EnergyScore_combined": energy_score_full.item(),
    "VariogramScore_combined": variogram_score_full.item(),
    "CRPS_2m_temperature": crps_per_var[0].item(),
    "CRPS_10m_windspeed": crps_per_var[1].item(),
    "EnergyScore_2m_temperature": energy_score_per_var[0].item(),
    "EnergyScore_10m_windspeed": energy_score_per_var[1].item(),
    "VariogramScore_2m_temperature": variogram_score_per_var[0].item(),
    "VariogramScore_10m_windspeed": variogram_score_per_var[1].item(),
}


update_wandb_run(f"feik/genpp/{MODEL_ID}", {"val": val_scores})

In [14]:
val_scores

{'CRPS_combined': 0.5469716191291809,
 'EnergyScore_combined': 33.66170120239258,
 'VariogramScore_combined': 461205.1875,
 'CRPS_2m_temperature': 0.6252567768096924,
 'CRPS_10m_windspeed': 0.46868592500686646,
 'EnergyScore_2m_temperature': 26.83710479736328,
 'EnergyScore_10m_windspeed': 20.489116668701172,
 'VariogramScore_2m_temperature': 262446.8125,
 'VariogramScore_10m_windspeed': 194044.5}